# Annual change in Thailand's international tourism receipts from a theme park

**Purpose.** This notebook applies the route rule to choose between the importance-weighted optimal-transport route and the ambient fallback route, and then reports the annual change in international tourism receipts that the opening of a park of the proposed cost would bring to the target economy. The route decision is stated first, before any estimate for the target economy is shown. The rule compares leave-one-economy-out prediction errors in which the importance weights are relearned inside every fold, requires a usable importance model and enough episodes, and requires that the target lies inside the support of the sources. The change is expressed in percentage points of GDP and in percent of the counterfactual receipts, scaled to US dollars and baht with the baseline of 2019 and of 2024 for every scenario of the proposal, and accompanied by the minimum detectable effect, the probability of detection and a reference comparison with the claims that officials have made. The description of the target economy is read from the previous notebook.

**Input files.** In the results directory: case_effects.csv, case_features.csv, case_features_imputed.csv, importance_diagnostics.json, loco_summary.csv, loco_table.csv, loco_fold_weights.csv, transport_usage.csv, transport_sensitivity.csv, thailand_target_baseline.csv, thailand_transport.json, thailand_transport_draws.csv, thailand_scenario_draws.csv and thailand_null_draws.csv. The run context also supplies the proposal file.

**Output files.** In the results directory: route_decision.json, ambient_loeo.csv, thailand_impact.csv and thailand_summary.json. In the figures directory: ot_predictive_distribution.png, impact_routes.png and impact_scaling_law.png.

**How to run.** On the desktop, set the environment variable DTT_RUN_REAL=1 and execute the notebook with `jupyter nbconvert --to notebook --execute --output-dir executed notebooks/05_thailand_impact.ipynb`. The simulated check sets DTT_WORLD=sim together with DTT_SIM_DIR, DTT_RESULTS_DIR and DTT_FIGURE_DIR, which name empty temporary directories, and needs the earlier notebooks to have run in the same mode.

**Run time.** About 10 to 20 seconds in the simulated mode and about 1 minute on the real data.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
_root = next(p for p in (_here, *_here.parents) if (p / "src" / "dtt" / "__init__.py").is_file())
sys.path.insert(0, str(_root / "src"))

from dtt import workflow

ctx = workflow.setup_run("05_thailand_impact")
ctx.describe()

## Route decision

The primary route is chosen before any estimate for the target economy is shown. The rule is applied without change: the primary route is the importance-weighted optimal-transport route if and only if four criteria hold. The importance diagnostics report a usable model. The leave-one-economy-out RMSE of ot_weighted is at most the stated multiple of that of equal and of ot_uniform. The validated sample has at least the stated number of episodes. The support check of the previous notebook reports that the target lies inside the support of the sources, because a transport of past effects cannot reach a target that is costlier or more concentrated than every case. In the validation the importance weights are relearned inside every fold from the training economies only, so the RMSE of ot_weighted does not use the left-out economy to choose its weights. The limits are printed below. A hash of the rule is printed and saved with the decision, so that the rule that was applied can be compared with the rule that was written down in advance. In every other case the primary route is the ambient route. The comparison of the transport with the ambient estimators and with the nearest-neighbour predictors is reported next to the decision and is not a criterion. The importance route and the ambient routes are reported side by side whichever route is primary.

In [ ]:
import hashlib
import json
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from scipy import stats

from dtt import impact, meta

NOTEBOOK_START = time.perf_counter()
ctx.require(
    "case_effects",
    "case_features",
    "case_features_imputed",
    "importance_diagnostics.json",
    "loco_summary",
    "loco_table",
    "loco_fold_weights",
    "transport_usage",
    "transport_sensitivity",
    "thailand_target_baseline",
    "thailand_transport.json",
    "thailand_transport_draws",
    "thailand_scenario_draws",
    "thailand_null_draws",
)
transport_record = ctx.read_json("thailand_transport")
for name in ("case_effects", "case_features_imputed"):
    if hashlib.sha256(ctx.table_path(name).read_bytes()).hexdigest() != transport_record["input_digests"][name]:
        raise RuntimeError(
            f"{ctx.table_path(name).name} differs from the file that 04_optimal_transport read; run notebook 04 again before this notebook"
        )
IMPORTANCE_KEY = "att_pp|primary"
importance_all = ctx.read_json("importance_diagnostics")
if IMPORTANCE_KEY not in importance_all:
    raise KeyError(f"importance_diagnostics.json has no entry {IMPORTANCE_KEY!r}; its entries are {sorted(importance_all)}")
importance_diagnostics = importance_all[IMPORTANCE_KEY]
IMPORTANCE_USABLE = bool(importance_diagnostics["usable"])
loco_summary = ctx.read_table("loco_summary", index_col="method")
loco_table = ctx.read_table("loco_table")
fold_weights = ctx.read_table("loco_fold_weights")
effects = ctx.read_table("case_effects")
n_primary = int(effects["sample_primary"].astype(bool).sum())
validated = loco_table[loco_table["method"] == "ot_weighted"]
n_validated = int(validated["case"].nunique())
n_validated_economies = int(validated["group"].nunique())
fold_fitted = fold_weights.groupby("held_out_group")["fitted"].all()
n_folds = int(fold_fitted.size)
n_uniform_folds = int((~fold_fitted).sum())
rules = dict(impact.DEFAULT_ROUTE_RULES)
support = transport_record["support"]

decision = impact.decide_route({"usable": IMPORTANCE_USABLE}, loco_summary, n_validated, support=support)

CRITERIA = ["importance_usable", "loco_rmse", "enough_cases", "target_supported"]
RULE = {
    "criteria": CRITERIA,
    "rmse_ratio_equal": rules["rmse_ratio_equal"],
    "rmse_ratio_uniform": rules["rmse_ratio_uniform"],
    "min_cases": rules["min_cases"],
    "support": support["settings"],
}
RULE_SHA256 = hashlib.sha256(json.dumps(RULE, sort_keys=True).encode("utf-8")).hexdigest()

print("Route rule, applied without change")
print(f"  SHA-256 of the rule: {RULE_SHA256}")
print("  The primary route is ot_importance if and only if all four criteria hold, and ambient otherwise.")
print("  1. The importance diagnostics report a usable model.")
print(
    f"  2. The leave-one-economy-out RMSE of ot_weighted, with the importance weights relearned inside every fold, is at most {rules['rmse_ratio_equal']:g} times that of equal "
    f"and at most {rules['rmse_ratio_uniform']:g} times that of ot_uniform."
)
print(f"  3. The validated sample has at least {rules['min_cases']:g} episodes; the independent units are the economies, and the validation leaves out whole economies.")
print(
    f"  4. The support check of notebook 04 reports that the target is supported: at least {support['settings']['min_ess']:g} effective sources, and no feature with an importance weight "
    f"of at least {support['settings']['min_weight']:g} outside the range of the sources that send mass, widened by {support['settings']['range_tolerance']:g} of that range."
)
print(f"\nInputs: importance entry {IMPORTANCE_KEY!r} with usable = {IMPORTANCE_USABLE}")
for reason in importance_diagnostics.get("reasons", []):
    print(f"  importance diagnostics: {reason}")
print(
    f"Primary episodes flagged: {n_primary}; validated: {n_validated} episodes in {n_validated_economies} economies; "
    f"importance weights relearned in {n_folds - n_uniform_folds} of {n_folds} folds and uniform in {n_uniform_folds}"
)
print("Leave-one-economy-out RMSE (percentage points of GDP):")
print(loco_summary.loc[["ot_weighted", "ot_uniform", "equal"], ["rmse"]].round(4).to_string())
print(
    f"Support of the target: effective number of sources {support['ess']:.2f}; largest share of the mass of one source {100 * support['max_source_share']:.1f} percent; "
    f"share of the importance weight on features outside the band {support['weighted_outside_share']:.3f}"
)
print("\nCriteria met:", {name: bool(flag) for name, flag in decision.passed.items()})
print(decision.details)
print(f"\nPrimary route: {decision.primary}")

route_record = {
    "primary": decision.primary,
    "passed": {name: bool(flag) for name, flag in decision.passed.items()},
    "details": decision.details,
    "rule": {
        "importance_usable_required": True,
        "rmse_ratio_equal": rules["rmse_ratio_equal"],
        "rmse_ratio_uniform": rules["rmse_ratio_uniform"],
        "min_cases": rules["min_cases"],
        "target_supported_required": True,
        "support_settings": support["settings"],
        "sha256": RULE_SHA256,
    },
    "inputs": {
        "importance_key": IMPORTANCE_KEY,
        "importance_usable": IMPORTANCE_USABLE,
        "importance_reasons": list(importance_diagnostics.get("reasons", [])),
        "loco_rmse": {str(method): float(value) for method, value in loco_summary["rmse"].items()},
        "n_primary_episodes": n_primary,
        "n_validated_episodes": n_validated,
        "n_validated_economies": n_validated_economies,
        "validation_weights": "importance relearned in every fold from the training economies",
        "n_folds": n_folds,
        "n_folds_uniform_weights": n_uniform_folds,
        "support": {
            "supported": bool(support["supported"]),
            "ess": float(support["ess"]),
            "max_source_share": float(support["max_source_share"]),
            "weighted_outside_share": float(support["weighted_outside_share"]),
            "reasons": list(support["reasons"]),
        },
    },
}
saved_route = ctx.save_json(route_record, "route_decision")

## Settings

The settings cell fixes every setting once. The flag FAST reduces only the number of random draws; the method is the same in every mode.

In [ ]:
# Settings (FAST changes the number of draws only)
FAST = ctx.mode == "sim"
N_DRAWS = 20000 if FAST else 100000  # predictive draws of each ambient estimator
N_DRAWS_FOLD = 4000 if FAST else 20000  # posterior draws in each leave-one-economy-out refit of A1
BASELINE_YEARS = (2019, 2024)  # baseline years that scale the effect to US dollars
PROBS = (0.05, 0.25, 0.5, 0.75, 0.95)  # quantile levels of the reported intervals
LEVEL = 0.9  # coverage of the Student prediction intervals of A2 and A3, and level of the conformal interval
RIDGE = 1.0  # ridge weight of A3, in units of the mean precision of one case on the standardised slopes
SELECTION_RATIO = 0.95  # A2 or A3 replaces A1 only if its RMSE is at most this multiple of the RMSE of A1
CLOSE_WITHIN = 0.05  # relative distance within which the selection of the ambient estimator is called close
ALPHA = 0.05  # size of the detection threshold
POWER = 0.8  # target power of the minimum detectable effect
A3_FEATURES = ["capex_pct_gdp", "receipts_pct_gdp"]  # covariates of the meta-regression
SEED_META = ctx.seeds["meta"]

print(f"FAST = {FAST}")
print(f"Draws: {N_DRAWS} per ambient estimator, {N_DRAWS_FOLD} per refit of A1 in the validation; seed {SEED_META}")
print(f"Baseline years {BASELINE_YEARS}; quantile levels {PROBS}")
print(f"A3 covariates {A3_FEATURES} with ridge weight {RIDGE:g}; A1 with the default priors of the module; prediction level {LEVEL:g}")
print(f"Selection of the ambient estimator is close within {CLOSE_WITHIN:g} of the RMSE or of the qualification limit")
print(f"Detection: size {ALPHA}, power {POWER}")

## Helper functions

In [ ]:
def show(table: pd.DataFrame, digits: int = 3) -> None:
    """Print a table with a fixed number of decimals."""
    print(table.round(digits).to_string())


def quantile_dict(values: np.ndarray, probs: tuple[float, ...] = PROBS) -> dict[str, float]:
    """Quantiles of a sample as a dictionary with the keys ``q5``, ``q25`` and so on."""
    return {f"q{round(100 * p):g}": float(q) for p, q in zip(probs, np.quantile(values, probs))}


def student_draws(prediction: dict[str, float], df: int, n: int, rng: np.random.Generator, label: str = "") -> np.ndarray | None:
    """Draws of the predictive distribution of a regression estimator.

    Parameters
    ----------
    prediction : dict
        Entries ``mean``, ``lo``, ``hi`` and ``se_pred`` of the prediction for the target.
    df : int
        Degrees of freedom of the Student distribution.
    n : int
        Number of draws.
    rng : numpy.random.Generator
        Random generator.
    label : str
        Name of the estimator, used in the message when its interval is unavailable.

    Returns
    -------
    numpy.ndarray or None
        ``n`` draws with location ``mean``, scale ``se_pred`` and ``df`` degrees of freedom. ``None``, after a printed
        message, when ``df`` is below one or a limit of the prediction interval is not finite, because the interval of
        the estimator is then unavailable and the estimator is left out of the draws.
    """
    limits = np.array([prediction["lo"], prediction["hi"]], dtype=float)
    if df < 1 or not np.isfinite(limits).all():
        print(f"{label}: the prediction interval has {df} degrees of freedom and no finite limits, so it is unavailable and the estimator is left out of the draws")
        return None
    return float(prediction["mean"]) + float(prediction["se_pred"]) * rng.standard_t(df, size=n)


def predict_held_out(estimator: str, train: pd.DataFrame, test: pd.DataFrame, outcome: str, se_column: str) -> np.ndarray:
    """Point predictions of held-out episodes by one ambient estimator.

    Parameters
    ----------
    estimator : str
        ``A1`` (pooled predictive mean), ``A2`` (scaling law in the cost share of GDP) or ``A3`` (meta-regression).
    train : pandas.DataFrame
        Episodes that the estimator is fitted to.
    test : pandas.DataFrame
        Episodes to predict.
    outcome : str
        Column of the effect.
    se_column : str
        Column of the standard error of the effect.

    Returns
    -------
    numpy.ndarray
        One prediction per row of ``test``.
    """
    y = train[outcome].to_numpy(dtype=float)
    s = train[se_column].to_numpy(dtype=float)
    with warnings.catch_warnings():
        # The refits are used for their point predictions; the interval warnings of small training samples are not shown
        warnings.filterwarnings("ignore", message="The prediction interval has")
        if estimator == "A1":
            fit = meta.bayes_normal_hierarchical(y, s, n_draws=N_DRAWS_FOLD, seed=SEED_META)
            return np.full(len(test), float(fit.summary.loc["predictive", "mean"]))
        if estimator == "A2":
            fit = meta.scaling_law(y, s, train["capex_pct_gdp"], test[["capex_pct_gdp"]], level=LEVEL)
            return np.asarray(fit["prediction"]["mean"], dtype=float)
        if estimator == "A3":
            fit = meta.meta_regression(y, s, train[A3_FEATURES], x_new=test[A3_FEATURES], ridge=RIDGE, level=LEVEL)
            return np.asarray(fit["prediction"]["mean"], dtype=float)
    raise ValueError(f"unknown estimator {estimator!r}")


def fit_ambient(estimator: str, frame: pd.DataFrame, outcome: str, se_column: str, target: dict[str, float], seed: int, label: str = "") -> dict[str, object]:
    """Fit one ambient estimator to the primary sample and draw the predictive distribution for the target.

    Parameters
    ----------
    estimator : str
        ``A1``, ``A2`` or ``A3``.
    frame : pandas.DataFrame
        Primary episodes with the effect, its standard error and the covariates.
    outcome : str
        Column of the effect.
    se_column : str
        Column of the standard error of the effect.
    target : dict
        Covariates of the target economy, with the names of ``A3_FEATURES``; the module matches them to the covariates
        of the episodes by name.
    seed : int
        Seed of the random draws.
    label : str
        Name of the fit, used in the message when the interval is unavailable.

    Returns
    -------
    dict
        ``draws`` (predictive draws, ``None`` when the interval of the estimator is unavailable), ``mean``, ``lo`` and
        ``hi`` (point prediction and interval at level ``LEVEL`` for A2 and A3, quantiles of the draws for A1) and
        ``fit`` (the output of the module function).
    """
    y = frame[outcome].to_numpy(dtype=float)
    s = frame[se_column].to_numpy(dtype=float)
    rng = np.random.default_rng(seed)
    if estimator == "A1":
        fit = meta.bayes_normal_hierarchical(y, s, n_draws=N_DRAWS, seed=seed)
        draws = fit.draws_pred
        lo, hi = np.quantile(draws, [0.5 - LEVEL / 2.0, 0.5 + LEVEL / 2.0])
        return {"draws": draws, "mean": float(draws.mean()), "lo": float(lo), "hi": float(hi), "fit": fit}
    if estimator == "A2":
        fit = meta.scaling_law(y, s, frame["capex_pct_gdp"], target, level=LEVEL)
    elif estimator == "A3":
        fit = meta.meta_regression(y, s, frame[A3_FEATURES], x_new=target, ridge=RIDGE, level=LEVEL)
    else:
        raise ValueError(f"unknown estimator {estimator!r}")
    prediction = fit["prediction"]
    draws = student_draws(prediction, int(fit["df"]), N_DRAWS, rng, label)
    return {"draws": draws, "mean": float(prediction["mean"]), "lo": float(prediction["lo"]), "hi": float(prediction["hi"]), "fit": fit}


def scaled_summary(draws_pp: np.ndarray, draws_rel: np.ndarray, row: pd.Series) -> dict[str, dict[str, dict[str, float]]]:
    """Quantiles of the scaled effect of one baseline year, for both scalings.

    Parameters
    ----------
    draws_pp : numpy.ndarray
        Draws of the effect in percentage points of GDP.
    draws_rel : numpy.ndarray
        Draws of the effect in percent of the counterfactual receipts.
    row : pandas.Series
        Baseline year with ``gdp_usd_bn``, ``receipts_usd_bn`` and ``fx_thb_per_usd``.

    Returns
    -------
    dict
        For each scaling (``absolute``, ``relative``) the quantiles ``q5``, ``q50`` and ``q95`` of the annual change
        in US$ billion (``usd_bn``), in THB billion (``thb_bn``), as a share of the baseline receipts
        (``share_of_receipts_pct``) and as a share of the baseline GDP (``share_of_gdp_pct``).
    """
    scaled = impact.scale_effect(draws_pp, draws_rel, row)
    out: dict[str, dict[str, dict[str, float]]] = {}
    for scaling, usd, thb in (
        ("absolute", scaled["absolute_usd_bn"], scaled["absolute_thb_bn"]),
        ("relative", scaled["relative_usd_bn"], scaled["relative_thb_bn"]),
    ):
        levels = (0.05, 0.5, 0.95)
        q_usd = np.quantile(usd, levels)
        q_thb = np.quantile(thb, levels)
        keys = ("q5", "q50", "q95")
        out[scaling] = {
            "usd_bn": dict(zip(keys, map(float, q_usd))),
            "thb_bn": dict(zip(keys, map(float, q_thb))),
            "share_of_receipts_pct": dict(zip(keys, (float(100.0 * q / row["receipts_usd_bn"]) for q in q_usd))),
            "share_of_gdp_pct": dict(zip(keys, (float(100.0 * q / row["gdp_usd_bn"]) for q in q_usd))),
        }
    return out

In [ ]:
UNIT = {"pp": "percentage points of GDP", "rel": "percent of the counterfactual receipts"}
SCALING_OF = {"pp": "absolute", "rel": "relative"}
BASE_COLUMN = {"pp": "gdp_usd_bn", "rel": "receipts_usd_bn"}
RECORD_KEY = {"pp": "att_pp", "rel": "att_rel_pct"}
LEVEL_TEXT = f"{100 * LEVEL:.0f} percent"
assert abs(float(transport_record["predictive"]["level"]) - LEVEL) < 1e-12, "the conformal level of notebook 04 differs from the level of this notebook"


def interval_bounds(draws: np.ndarray, route: str, record: dict[str, object] | None) -> tuple[float, float, str]:
    """Interval of an effect that is quoted for a route.

    Parameters
    ----------
    draws : numpy.ndarray
        Draws of the route in the unit of the effect.
    route : str
        Key of the route. The importance route (``ot_importance``) is quoted with its conformal interval.
    record : dict or None
        Conformal interval of the effect in the unit of the draws, as saved by notebook 04.

    Returns
    -------
    lower, upper : float
        Bounds of the interval at the level ``LEVEL``.
    kind : str
        Which of the intervals the bounds are: the conformal interval, the percentile interval of the bootstrap draws of
        the transported average effect, or the percentile interval of the predictive draws.
    """
    if route == "ot_importance":
        if record is not None and record["status"] == "ok":
            return float(record["lower"]), float(record["upper"]), "conformal interval of the effect observed in a new study"
        kind = "percentile interval of the predictive draws (no conformal interval exists)"
    elif route == "ot_importance_sampling":
        kind = "percentile interval of the bootstrap draws of the transported average effect"
    else:
        kind = "percentile interval of the predictive draws of the ambient estimator"
    lower, upper = np.quantile(draws, [0.5 - LEVEL / 2.0, 0.5 + LEVEL / 2.0])
    return float(lower), float(upper), kind


def central_value(draws: np.ndarray, route: str, estimate: float | None, record: dict[str, object] | None) -> tuple[float, str]:
    """Central value that is quoted with the interval of a route.

    Parameters
    ----------
    draws : numpy.ndarray
        Draws of the route in the unit of the effect.
    route : str
        Key of the route.
    estimate : float or None
        Transported effect in the unit of the draws; used for the importance route only.
    record : dict or None
        Conformal interval of the effect, as in ``interval_bounds``.

    Returns
    -------
    value : float
        The transported effect for the importance route when its conformal interval exists, because that interval is
        symmetric about the transported effect, and the median of the draws otherwise.
    kind : str
        ``transported effect`` or ``median of the draws``.
    """
    if route == "ot_importance" and estimate is not None and record is not None and record["status"] == "ok":
        return float(estimate), "transported effect"
    return float(np.median(draws)), "median of the draws"

## Target economy and baseline

The target economy is read from the description that the previous notebook saved: the scenarios of the proposal with their cost as a share of GDP, the receipts share of GDP of the target and the baseline put on the basis of the panel series. The cost share divides the cost in US dollars at the exchange rate of the cost year by the GDP of that year; the primary reading uses the latest baseline year, and the alternative reading uses another year that the previous notebook names. The receipts share is the mean of the last years of the target cloud on the basis of the panel, which is the basis on which the effects of the cases are measured. The baseline years used for the scaling are printed with the receipts that the scaling multiplies, the receipts of the baseline file, the basis of each year and the ratio of the panel to the baseline where a year has no panel value.

In [ ]:
target_record = transport_record["target"]
baseline = ctx.read_table("thailand_target_baseline", index_col="year")
lacking_years = [year for year in BASELINE_YEARS if year not in baseline.index]
assert not lacking_years, f"the target baseline has no row for {lacking_years}"
scenario = ctx.primary_scenario()
assert scenario["name"] == target_record["scenario"] and abs(float(scenario["capex_thb_bn"]) - target_record["capex_thb_bn"]) < 1e-9, "the proposal differs from the one used by notebook 04"
SCENARIOS = transport_record["scenarios"]
PRIMARY_SCENARIO = str(target_record["scenario"])
SCEN_ORDER = [PRIMARY_SCENARIO, *sorted((n for n in SCENARIOS if n != PRIMARY_SCENARIO), key=lambda n: -SCENARIOS[n]["capex_thb_bn"])]
TARGET = {"capex_pct_gdp": float(target_record["capex_pct_gdp"]), "receipts_pct_gdp": float(target_record["receipts_pct_gdp"])}
TARGET_ALTERNATIVE = {"capex_pct_gdp": float(target_record["capex_pct_gdp_alternative"]), "receipts_pct_gdp": TARGET["receipts_pct_gdp"]}
COST_YEAR = int(target_record["cost_year"])
COST_YEAR_ALTERNATIVE = int(target_record["cost_year_alternative"])

print(f"Target economy: {ctx.target_name}; proposal status: {ctx.proposal.get('status', 'not stated')}")
print(
    f"Primary scenario {PRIMARY_SCENARIO}: THB {target_record['capex_thb_bn']:g} billion at {target_record['fx_thb_per_usd']:.3f} THB per US$ ({COST_YEAR}) = "
    f"US$ {target_record['capex_usd_bn']:.3f} billion, {TARGET['capex_pct_gdp']:.3f} percent of the GDP of {COST_YEAR}; with the cost year {COST_YEAR_ALTERNATIVE} the share is {TARGET_ALTERNATIVE['capex_pct_gdp']:.3f} percent"
)
print(f"Receipts as a share of GDP on the basis of the panel, mean of {target_record['receipts_window_years'][0]} to {target_record['receipts_window_years'][-1]}: {TARGET['receipts_pct_gdp']:.3f} percent")
scenario_frame = pd.DataFrame(target_record["scenarios"]).T.loc[SCEN_ORDER, ["capex_thb_bn", "capex_usd_bn", "capex_pct_gdp", "capex_pct_gdp_alternative"]].astype(float)
print("\nScenarios of the proposal")
show(scenario_frame, 3)
print("\nBaseline years used for the scaling")
show(baseline.loc[list(BASELINE_YEARS), ["gdp_usd_bn", "receipts_usd_bn_baseline", "receipts_usd_bn", "receipts_basis", "basis_ratio", "receipts_pct_gdp", "fx_thb_per_usd"]], 3)

## Importance route

The importance route is shown here for the first time with its effects. Its predictive distribution is built from the leave-one-economy-out prediction errors of the transport rule that the previous notebook validated. Each draw is the transported effect minus one such error, with the economies resampled, and the draws are rescaled about the transported effect so that their symmetric interval at the stated level equals the conformal interval. The predictive distribution describes the effect that a new study of an economy outside the sample would observe, including the systematic bias and the dispersion of the rule across economies. It does not use the uncertainty of the transported mean.

Three intervals are printed and each answers its own question. The conformal interval is the predictive interval for the effect that a new study at the target would observe, and it is the interval that is quoted for the importance route. The bootstrap interval of the transported average effect is the sampling uncertainty of the mean effect and is shown as a sensitivity row of the route tables. The bootstrap predictive percentiles are kept as a further sensitivity and have no coverage guarantee. The coverage statement of the conformal interval holds for a target that resembles the cases, which the support result printed with the decision reports.

In [ ]:
transport_draws = ctx.read_table("thailand_transport_draws")
scenario_draws = ctx.read_table("thailand_scenario_draws")
null_draws = ctx.read_table("thailand_null_draws")
transport_sensitivity = ctx.read_table("transport_sensitivity")
PREDICTIVE_METHOD = str(transport_record["predictive"]["method"])
OT_RECORD = {"pp": transport_record["att_pp"], "rel": transport_record["att_rel_pct"]}
OT_SAMPLING = {"pp": transport_draws["transport_pp"].to_numpy(dtype=float), "rel": transport_draws["transport_rel_pct"].to_numpy(dtype=float)}
print(
    f"Predictive draws of the importance route: {len(transport_draws)}, from the leave-one-economy-out errors of {PREDICTIVE_METHOD} "
    f"({transport_record['predictive']['n_episodes']} episodes in {transport_record['predictive']['n_economies']} economies); null draws: {len(null_draws)}"
)
for outcome in ("pp", "rel"):
    r = OT_RECORD[outcome]
    ci = r["conformal_interval"]
    print(f"\nEffect in {UNIT[outcome]}")
    print(f"  Transported effect (point estimate): {r['estimate']:.4f}")
    if ci["status"] == "ok":
        print(
            f"  Conformal {LEVEL_TEXT} interval (effect observed in a new study at the target): {ci['lower']:.4f} to {ci['upper']:.4f}; "
            f"guaranteed coverage {ci['guaranteed_level']:.3f} from {ci['n']} validation errors, order statistic {ci['rank']}"
        )
        print(
            f"  Support of the target by the sources (notebook 04): {'supported' if support['supported'] else 'not supported'}; "
            f"effective number of sources {support['ess']:.2f}, limit {support['settings']['min_ess']:g}"
        )
    else:
        print(f"  No conformal interval: {ci['status']}")
    b, sp, pr = r["bootstrap_percentiles"], r["sampling_predictive_percentiles"], r["predictive_percentiles"]
    print(f"  Bootstrap interval of the transported average effect (sampling uncertainty), 5th to 95th percentile: {b['p5']:.4f} to {b['p95']:.4f}")
    print(f"  Bootstrap predictive percentiles (no coverage guarantee), 5th to 95th: {sp['p5']:.4f} to {sp['p95']:.4f}")
    print(f"  Descriptive percentiles of the saved predictive draws, 5th to 95th: {pr['p5']:.4f} to {pr['p95']:.4f}")
by_year = pd.DataFrame({"att_pp": OT_RECORD["pp"]["by_target_year"], "att_rel_pct": OT_RECORD["rel"]["by_target_year"]})
print("\nTransported effect by target year")
show(by_year, 4)
print("\nTransported effect of every configuration of the previous notebook (point estimates)")
show(transport_sensitivity[["sample", "target_cloud", "weights", "rho_source", "n_sources", "ess", "att_pp", "att_rel_pct", "converged"]], 4)
print(
    f"Range over the {len(transport_sensitivity)} configurations: att_pp from {transport_sensitivity['att_pp'].min():.4f} to {transport_sensitivity['att_pp'].max():.4f} percentage points of GDP; "
    f"att_rel_pct from {transport_sensitivity['att_rel_pct'].min():.2f} to {transport_sensitivity['att_rel_pct'].max():.2f} percent; base configuration {OT_RECORD['pp']['estimate']:.4f} and {OT_RECORD['rel']['estimate']:.2f}"
)
print(f"Configurations that did not converge: {int((~transport_sensitivity['converged']).sum())}")

## Ambient routes

The ambient routes use the primary episodes that have a value for every covariate, and the placebo standard deviation of each effect as its standard error. Imputed covariate values are taken from the imputed feature table, and a covariate that the imputation dropped is taken from the observed feature table. A1 is the Bayesian normal hierarchical model, whose posterior predictive distribution is the pooled effect of a new opening; its mean effect has the default normal prior of the module, centred at zero and wide on the scale of the data, and the prior in use is printed with the fit. A2 is the scaling law through the origin in the cost as a share of GDP. A3 is the mixed-effects meta-regression on the cost share and the receipts share, with a ridge penalty on the standardised slopes. The two regression routes give Student predictive distributions whose degrees of freedom are printed with each fit; when they are below one the interval of the route is unavailable, a message says so and the route is left out of the draws. The covariates of the target are matched to those of the episodes by name.

A regression route extrapolates the fitted line when a covariate of the target lies outside the range of the episodes. The position of the target against that range is printed for each covariate, together with the ratio of the target value to the largest value among the episodes, because the ridge penalty on the slopes moves an extrapolated prediction more than one inside the range.

In [ ]:
features_observed = ctx.read_table("case_features").set_index("case_id")
features_imputed = ctx.read_table("case_features_imputed").set_index("case_id")
frame = effects.copy()
for name in A3_FEATURES:
    filled = features_imputed if name in features_imputed.columns else features_observed
    frame[f"{name}_observed"] = frame["case_id"].map(features_observed[name])
    frame[name] = frame["case_id"].map(filled[name])
    print(f"Covariate {name}: taken from the {'imputed' if name in features_imputed.columns else 'observed'} feature table")
NEEDED = ["att_pp", "att_rel_pct", "placebo_sd_pp", "placebo_sd_rel_pct", *A3_FEATURES]
flagged = frame[frame["sample_primary"].astype(bool)]
AMB = flagged[np.isfinite(flagged[NEEDED].to_numpy(dtype=float)).all(axis=1)].reset_index(drop=True)
assert len(AMB) >= 6, f"the ambient routes need at least six primary episodes with complete covariates; found {len(AMB)}"
assert (AMB["placebo_sd_pp"] > 0).all() and (AMB["placebo_sd_rel_pct"] > 0).all(), "standard errors must be positive"
GROUPS = AMB["economy_group"].to_numpy()
n_imputed = {name: int(AMB[f"{name}_observed"].isna().sum()) for name in A3_FEATURES}
print(f"Primary episodes flagged: {len(flagged)}; with complete values for the ambient routes: {len(AMB)} in {len(set(GROUPS))} economies; values imputed: {n_imputed}")
print(f"Cost share of GDP in the sample: from {AMB['capex_pct_gdp'].min():.3f} to {AMB['capex_pct_gdp'].max():.3f} percent; target {TARGET['capex_pct_gdp']:.3f} percent, {TARGET['capex_pct_gdp'] / AMB['capex_pct_gdp'].max():.2f} times the largest case")
print(f"Receipts share of GDP in the sample: from {AMB['receipts_pct_gdp'].min():.3f} to {AMB['receipts_pct_gdp'].max():.3f} percent; target {TARGET['receipts_pct_gdp']:.3f} percent")
extrapolation_table = meta.extrapolation(AMB[A3_FEATURES], TARGET)
print("\nPosition of the target against the range of the episodes of the ambient routes")
show(extrapolation_table, 3)
EXTRAPOLATES = extrapolation_table.index[extrapolation_table["position"] != "inside"].tolist()
print("Covariates on which the regression routes extrapolate:", EXTRAPOLATES or "none")

### Selection of the ambient estimator

The selection rule is fixed before the numbers below are computed. Each estimator is refitted without every episode of the held-out economy and predicts the held-out episodes, and the RMSE of these predictions is compared. A2 or A3 becomes the ambient estimator only if its RMSE is below the RMSE of A1 by the margin printed below; if both qualify, the one with the smaller RMSE is chosen; otherwise A1 is the ambient estimator. The comparison uses the effect in percentage points of GDP and the selected estimator is applied to both outcomes. The selection is reported with its margin: the runner up is the best estimator that was not selected, and the selection is called close when the runner up lies within the relative distance printed in the settings of the selected RMSE or when an estimator lies within that distance of the qualification limit, so that a small change in the RMSE values could change the choice.

The leave-one-economy-out RMSE of the ambient estimators is then passed to the route rule again, together with the sizes of the two validation samples. The comparison of ot_weighted with the best ambient estimator and with the best nearest-neighbour predictor is reported by the rule and is not a criterion; the primary route and the four criteria must be the same as in the decision above.

In [ ]:
ESTIMATORS = ("A1", "A2", "A3")
DESCRIPTION = {
    "A1": "hierarchical normal model, pooled posterior predictive",
    "A2": "scaling law in the cost share of GDP",
    "A3": "meta-regression on the cost share and the receipts share",
}
print(f"Rule: A2 or A3 replaces A1 only if its RMSE is at most {SELECTION_RATIO:g} times the RMSE of A1 (at least {100 * (1 - SELECTION_RATIO):.0f} percent lower)")
held_out = {name: np.full(len(AMB), np.nan) for name in ESTIMATORS}
for economy in sorted(set(GROUPS)):
    test_mask = GROUPS == economy
    assert int((~test_mask).sum()) >= 3, "too few episodes outside the held-out economy"
    for name in ESTIMATORS:
        held_out[name][test_mask] = predict_held_out(name, AMB[~test_mask], AMB[test_mask], "att_pp", "placebo_sd_pp")
    if ctx.guard:
        ctx.guard()
observed = AMB["att_pp"].to_numpy(dtype=float)
loeo = pd.DataFrame(
    {
        "estimator": list(ESTIMATORS),
        "description": [DESCRIPTION[name] for name in ESTIMATORS],
        "rmse": [float(np.sqrt(np.mean((observed - held_out[name]) ** 2))) for name in ESTIMATORS],
        "mae": [float(np.mean(np.abs(observed - held_out[name]))) for name in ESTIMATORS],
        "mean_error": [float(np.mean(observed - held_out[name])) for name in ESTIMATORS],
    }
).set_index("estimator", drop=False)
selection = impact.select_ambient({name: float(loeo.loc[name, "rmse"]) for name in ESTIMATORS}, baseline="A1", ratio=SELECTION_RATIO, close_within=CLOSE_WITHIN)
SELECTED = str(selection["selected"])
loeo["ratio_to_A1"] = [selection["ratios"][name] for name in ESTIMATORS]
loeo["qualifies"] = [name in selection["qualifying"] for name in ESTIMATORS]
loeo["selected"] = loeo.index == SELECTED
loeo["n_episodes"] = len(AMB)
loeo["n_economies"] = len(set(GROUPS))
show(loeo.drop(columns=["estimator"]), 4)
print(f"\nAmbient estimator selected by the rule: {SELECTED} ({DESCRIPTION[SELECTED]})")
print(
    f"Qualifying estimators: {selection['qualifying'] or 'none'}; runner up {selection['runner_up']} with an RMSE {100 * selection['margin']:+.1f} percent relative to the selected estimator; "
    f"selection close: {selection['close']}"
)
if selection["close"]:
    print("The selection is close, so a small change in the RMSE values could select another estimator; the other ambient estimators are reported next to the selected one.")
print(
    f"For reference, leave-one-economy-out RMSE of the transport predictors, with the importance weights relearned in every fold "
    f"({n_validated} validated episodes; {len(AMB)} episodes in the ambient validation):"
)
print(loco_summary["rmse"].round(4).to_string())
saved_loeo = ctx.save_table(loeo.reset_index(drop=True), "ambient_loeo")

ambient_rmse = {name: float(loeo.loc[name, "rmse"]) for name in ESTIMATORS}
decision_full = impact.decide_route(
    {"usable": IMPORTANCE_USABLE}, loco_summary, n_validated, support=support, ambient_rmse=ambient_rmse, n_ot=n_validated, n_ambient=len(AMB)
)
assert decision_full.primary == decision.primary and decision_full.passed == decision.passed, "the comparison with the ambient estimators changed the decision"
decision = decision_full
print("\nComparison reported next to the decision (it is not a criterion)")
for key, value in decision.info.items():
    print(f"  {key}: {value}")
print(decision.details)

route_record["details"] = decision.details
route_record["info"] = {key: value for key, value in decision.info.items()}
route_record["ambient_selection"] = {
    "rule": f"A2 or A3 replaces A1 only if its leave-one-economy-out RMSE is at most {SELECTION_RATIO:g} times that of A1; the smaller RMSE wins if both qualify",
    "selection_ratio": SELECTION_RATIO,
    "outcome": "att_pp",
    "rmse": ambient_rmse,
    "ratio_to_A1": {name: float(selection["ratios"][name]) for name in ESTIMATORS},
    "selected": SELECTED,
    "qualifying": list(selection["qualifying"]),
    "runner_up": selection["runner_up"],
    "margin": float(selection["margin"]),
    "close": bool(selection["close"]),
    "n_episodes": int(len(AMB)),
    "n_economies": int(len(set(GROUPS))),
}
saved_route = ctx.save_json(route_record, "route_decision")

### Predictions of the ambient estimators for the target

Every estimator is fitted to the whole primary sample for both outcomes and for every scenario of the proposal; A1 does not depend on the target and is fitted once. The covariates of the target are the cost share of the scenario and the receipts share printed above. The cost share of the alternative cost year is used for the regression routes as a sensitivity. The ridge penalty of A3 shrinks the standardised slopes towards zero; the table of slopes with and without the penalty shows how much, and for a target outside the range of the episodes the shrinkage moves the prediction.

In [ ]:
AMBIENT_BY_SCENARIO: dict[str, dict[str, dict[str, dict[str, object]]]] = {}
for k, scn in enumerate(SCEN_ORDER):
    target_s = {"capex_pct_gdp": float(SCENARIOS[scn]["capex_pct_gdp"]), "receipts_pct_gdp": TARGET["receipts_pct_gdp"]}
    block: dict[str, dict[str, dict[str, object]]] = {}
    for i, name in enumerate(ESTIMATORS):
        if name == "A1" and k > 0:
            block[name] = AMBIENT_BY_SCENARIO[PRIMARY_SCENARIO][name]
            continue
        block[name] = {
            "pp": fit_ambient(name, AMB, "att_pp", "placebo_sd_pp", target_s, SEED_META + 10 * i, label=f"{name}, scenario {scn}, effect in percentage points of GDP"),
            "rel": fit_ambient(name, AMB, "att_rel_pct", "placebo_sd_rel_pct", target_s, SEED_META + 10 * i + 1, label=f"{name}, scenario {scn}, effect in percent of the counterfactual receipts"),
        }
    AMBIENT_BY_SCENARIO[scn] = block
    if ctx.guard:
        ctx.guard()
AMBIENT = AMBIENT_BY_SCENARIO[PRIMARY_SCENARIO]
AVAILABLE = [name for name in ESTIMATORS if all(AMBIENT_BY_SCENARIO[scn][name][outcome]["draws"] is not None for scn in SCEN_ORDER for outcome in ("pp", "rel"))]
print(f"Estimators with a predictive interval: {AVAILABLE}; left out of the draws: {[name for name in ESTIMATORS if name not in AVAILABLE]}")
assert SELECTED in AVAILABLE, f"the selected ambient estimator {SELECTED} has no predictive interval"
prediction_rows = []
for name in AVAILABLE:
    for outcome, label in (("pp", "att_pp"), ("rel", "att_rel_pct")):
        result = AMBIENT[name][outcome]
        q = np.quantile(result["draws"], PROBS)
        prediction_rows.append({"estimator": name, "outcome": label, "mean": result["mean"], "q5": q[0], "q25": q[1], "q50": q[2], "q75": q[3], "q95": q[4]})
prediction_table = pd.DataFrame(prediction_rows)
print(f"Primary scenario {PRIMARY_SCENARIO}")
show(prediction_table, 4)

scenario_prediction = pd.DataFrame(
    [
        {"scenario": scn, "estimator": name, "mean_att_pp": AMBIENT_BY_SCENARIO[scn][name]["pp"]["mean"], "mean_att_rel_pct": AMBIENT_BY_SCENARIO[scn][name]["rel"]["mean"]}
        for scn in SCEN_ORDER
        for name in AVAILABLE
    ]
)
print("\nPoint predictions of the ambient estimators by scenario")
show(scenario_prediction, 4)

alternative_rows = []
for name in ("A2", "A3"):
    for outcome, column, se_column in (("pp", "att_pp", "placebo_sd_pp"), ("rel", "att_rel_pct", "placebo_sd_rel_pct")):
        alternative = fit_ambient(name, AMB, column, se_column, TARGET_ALTERNATIVE, SEED_META + 10 * ESTIMATORS.index(name), label=f"{name}, alternative cost year, {RECORD_KEY[outcome]}")
        alternative_rows.append(
            {
                "estimator": name,
                "outcome": RECORD_KEY[outcome],
                "mean_primary_cost_year": AMBIENT[name][outcome]["mean"],
                "mean_alternative_cost_year": alternative["mean"],
            }
        )
print(f"\nPoint predictions of the regression routes with the cost share of the year {COST_YEAR_ALTERNATIVE} ({TARGET_ALTERNATIVE['capex_pct_gdp']:.3f} percent) against the year {COST_YEAR} ({TARGET['capex_pct_gdp']:.3f} percent)")
show(pd.DataFrame(alternative_rows), 4)

a1_fit = AMBIENT["A1"]["pp"]["fit"]
a1_fit_rel = AMBIENT["A1"]["rel"]["fit"]
print("\nA1 on the effect in percentage points of GDP (posterior summary)")
show(a1_fit.summary, 4)
print(
    f"Prior of the mean effect: normal with mean {a1_fit.mu_prior[0]:g} and standard deviation {a1_fit.mu_prior[1]:.4f} (percentage points of GDP) and "
    f"{a1_fit_rel.mu_prior[1]:.4f} (percent of the counterfactual); prior of the between-episode standard deviation: half-normal with scale "
    f"{a1_fit.tau_prior_scale:.4f} and {a1_fit_rel.tau_prior_scale:.4f}"
)
print(f"Posterior probability that the predictive effect is positive: {a1_fit.prob_pred_positive:.3f}")
a2_fit = AMBIENT["A2"]["pp"]["fit"]
print(f"\nA2 on the effect in percentage points of GDP: slope {a2_fit['beta']:.4f} percentage points per percentage point of GDP of cost (standard error {a2_fit['se_beta']:.4f}); between-episode sd {np.sqrt(a2_fit['tau2']):.4f}; degrees of freedom {a2_fit['df']}")
a3_fit = AMBIENT["A3"]["pp"]["fit"]
print("\nA3 on the effect in percentage points of GDP: coefficients on standardised features")
show(a3_fit["coef"], 4)
print(f"Between-episode sd {np.sqrt(a3_fit['tau2']):.4f}; degrees of freedom {a3_fit['df']}")
print(f"Standardised slopes of A3 with the ridge weight {RIDGE:g} and without it; a ratio below one means that the penalty shrinks the slope")
show(a3_fit["shrinkage"], 4)

## Scaling to US dollars and baht

The absolute scaling multiplies the effect in percentage points of GDP by the GDP of the baseline year. The relative scaling multiplies the effect in percent of the counterfactual receipts by the receipts of the baseline year on the basis of the panel. Both give the annual change in receipts in US$ billion, and the exchange rate of the baseline year converts it to THB billion. Every scenario of the proposal is scaled, not only the primary one. The route table has one block for the importance route, one for its sampling-only sensitivity (primary scenario only), one for the selected ambient estimator under the name ambient, and one for each ambient estimator under its own name. The columns interval_lower and interval_upper hold the interval at the level printed in the settings and the column interval_kind says which interval it is: the conformal interval for the importance route, the percentile interval of the bootstrap draws for the sampling-only sensitivity, and the percentile interval of the predictive draws for the ambient routes. The columns q5 to q95 are percentiles of the draws of the route, and the column central is the value that is quoted with the interval: the transported effect for the importance route, because the conformal interval is symmetric about it, and the median of the draws for every other route; the column central_kind says which. The column importance_route_supported says whether the support check of the previous notebook reports the target of the scenario as supported by the sources.

In [ ]:
ROUTE_DRAWS: dict[str, dict[str, dict[str, np.ndarray]]] = {}
for scn in SCEN_ORDER:
    scenario_sample = scenario_draws[scenario_draws["scenario"] == scn].sort_values("draw")
    assert len(scenario_sample) == len(transport_draws), f"the draws of the scenario {scn} are incomplete"
    routes = {
        "ot_importance": {
            "pp": scenario_sample["predictive_pp"].to_numpy(dtype=float),
            "rel": scenario_sample["predictive_rel_pct"].to_numpy(dtype=float),
        }
    }
    if scn == PRIMARY_SCENARIO:
        routes["ot_importance_sampling"] = {"pp": OT_SAMPLING["pp"], "rel": OT_SAMPLING["rel"]}
    routes["ambient"] = {outcome: AMBIENT_BY_SCENARIO[scn][SELECTED][outcome]["draws"] for outcome in ("pp", "rel")}
    for name in AVAILABLE:
        routes[f"ambient_{name}"] = {outcome: AMBIENT_BY_SCENARIO[scn][name][outcome]["draws"] for outcome in ("pp", "rel")}
    ROUTE_DRAWS[scn] = routes
assert np.array_equal(ROUTE_DRAWS[PRIMARY_SCENARIO]["ot_importance"]["pp"], transport_draws["predictive_pp"].to_numpy(dtype=float))
ESTIMATOR_OF = {"ot_importance": "transport", "ot_importance_sampling": "transport_sampling", "ambient": SELECTED, **{f"ambient_{name}": name for name in AVAILABLE}}

parts = []
for scn in SCEN_ORDER:
    tables = {
        route: impact.impact_table(draws["pp"], draws["rel"], baseline, years=BASELINE_YEARS, route=route, probs=PROBS)
        for route, draws in ROUTE_DRAWS[scn].items()
    }
    part = impact.route_comparison(tables, decision)
    part.insert(0, "scenario", scn)
    parts.append(part)
comparison = pd.concat(parts, ignore_index=True)
comparison["estimator"] = comparison["route"].map(ESTIMATOR_OF)
comparison["capex_thb_bn"] = comparison["scenario"].map({n: float(SCENARIOS[n]["capex_thb_bn"]) for n in SCEN_ORDER})
comparison["capex_pct_gdp"] = comparison["scenario"].map({n: float(SCENARIOS[n]["capex_pct_gdp"]) for n in SCEN_ORDER})
comparison["importance_route_supported"] = comparison["scenario"].map({n: bool(SCENARIOS[n]["support"]["supported"]) for n in SCEN_ORDER})
fx_by_year = baseline["fx_thb_per_usd"]
bounds_rows = []
for _, row in comparison.iterrows():
    outcome = "pp" if row["scaling"] == "absolute" else "rel"
    scenario_entry = SCENARIOS[row["scenario"]][RECORD_KEY[outcome]]
    record = scenario_entry["conformal_interval"] if row["route"] == "ot_importance" else None
    draws_row = ROUTE_DRAWS[row["scenario"]][row["route"]][outcome]
    lower, upper, kind = interval_bounds(draws_row, row["route"], record)
    centre, centre_kind = central_value(draws_row, row["route"], scenario_entry["estimate"], record)
    factor = float(baseline.loc[row["baseline_year"], BASE_COLUMN[outcome]]) / 100.0
    bounds_rows.append((lower * factor, upper * factor, kind, centre * factor, centre_kind))
comparison["interval_lower"] = [b[0] for b in bounds_rows]
comparison["interval_upper"] = [b[1] for b in bounds_rows]
comparison["interval_kind"] = [b[2] for b in bounds_rows]
comparison["interval_level"] = LEVEL
comparison["central"] = [b[3] for b in bounds_rows]
comparison["central_kind"] = [b[4] for b in bounds_rows]
for column in ("q5", "q50", "q95", "interval_lower", "interval_upper", "central"):
    comparison[f"{column}_thb_bn"] = comparison[column] * comparison["baseline_year"].map(fx_by_year)

print(f"Annual change in international tourism receipts, US$ billion, primary scenario {PRIMARY_SCENARIO} (primary route: {decision.primary})")
show(comparison[comparison["scenario"] == PRIMARY_SCENARIO][["route", "estimator", "baseline_year", "scaling", "q5", "q25", "q50", "q75", "q95", "central", "interval_lower", "interval_upper", "median_share_receipts_pct", "median_share_gdp_pct", "primary", "importance_route_supported"]], 3)
print("\nKinds of interval in the columns interval_lower and interval_upper, and of the central value in the column central")
for route, group in comparison[comparison["scenario"] == PRIMARY_SCENARIO].groupby("route"):
    print(f"  {route}: interval is the {group['interval_kind'].iloc[0]}; central value is the {group['central_kind'].iloc[0]}")
print(f"\nPrimary route {decision.primary} in the other scenarios of the proposal, US$ billion")
others = comparison[(comparison["scenario"] != PRIMARY_SCENARIO) & comparison["primary"]]
show(others[["scenario", "capex_thb_bn", "capex_pct_gdp", "route", "baseline_year", "scaling", "central", "interval_lower", "interval_upper", "importance_route_supported"]], 3)
saved_impact = ctx.save_table(comparison, "thailand_impact")

## Detection

The transported placebo draws of the previous notebook describe what the method reports where nothing happened. The detection threshold is the quantile of their absolute values at one minus the size of the test. The minimum detectable effect is the threshold plus the larger of the placebo quantile at the power and the negative of the placebo quantile at one minus the power, so that an effect of that size and of either sign exceeds the threshold in about the stated share of the placebo draws, the shares being printed below because the empirical placebo distribution has repeated values and the two signs can differ slightly; the empirical placebo distribution is used as it is, and the normal approximation, the threshold plus the standard normal quantile of the power times the standard deviation of the placebo draws, is reported next to it.

The placebo effects in percentage points of GDP come from the economies of the sources, and the noise of such an effect is proportional to the receipts of that economy relative to its GDP. The null of the absolute scaling is therefore put on the scale of the target: its draws are multiplied by the receipts share of GDP of the target divided by the receipts share of the sources behind the target, weighted by the mass that each source sends under the plan. The null of the relative scaling is in percent of the counterfactual receipts, is free of the level of receipts, and is used as it is.

The predictive draws of the importance route are observed effects, because they carry the prediction errors of earlier openings, so the detection probability of that route is the share of its draws beyond the threshold. The draws of the ambient routes describe the true effect of a new opening, so each draw is observed with an independent draw from the placebo distribution before it is compared with the threshold. The detection probability of every scenario uses the same null, which belongs to the primary scenario. The size and the power are printed below.

In [ ]:
share_table = features_imputed if "receipts_pct_gdp" in features_imputed.columns else features_observed
usage_table = ctx.read_table("transport_usage").set_index("case_id")
source_shares = share_table["receipts_pct_gdp"].reindex(usage_table.index)
SOURCE_SHARE = impact.source_receipts_share(source_shares, usage_table["mass_sent"])
NULL_PP_RAW = null_draws["null_pp"].to_numpy(dtype=float)
NULL_PP, NULL_FACTOR = impact.rescale_null_pp(NULL_PP_RAW, SOURCE_SHARE, TARGET["receipts_pct_gdp"])
NULLS = {"pp": NULL_PP, "rel": null_draws["null_rel_pct"].to_numpy(dtype=float)}
print(
    f"Receipts share of GDP of the sources behind the target, weighted by the mass they send: {SOURCE_SHARE:.3f} percent; target: {TARGET['receipts_pct_gdp']:.3f} percent; "
    f"factor applied to the null in percentage points of GDP: {NULL_FACTOR:.3f}"
)
print(
    f"Standard deviation of the null in percentage points of GDP: {NULL_PP_RAW.std(ddof=1):.4f} before and {NULL_PP.std(ddof=1):.4f} after the rescaling; "
    f"standard deviation of the null in percent of the counterfactual receipts: {NULLS['rel'].std(ddof=1):.3f} (not rescaled)"
)
TRUE_EFFECT_ROUTES = ("ambient", *(f"ambient_{name}" for name in AVAILABLE))
DETECTION_ROUTES = [route for route in ROUTE_DRAWS[PRIMARY_SCENARIO] if route != "ot_importance_sampling"]
detection: dict[str, dict[str, float]] = {}
print(f"Size of the test {ALPHA}; power {POWER}; standard normal quantile of the power {stats.norm.ppf(POWER):.4f}")
for outcome in ("pp", "rel"):
    limits = impact.minimum_detectable_effect(NULLS[outcome], alpha=ALPHA, power=POWER)
    detection[outcome] = {
        "threshold": limits["threshold"],
        "mde": limits["mde"],
        "mde_normal": limits["mde_normal"],
        "null_sd": float(NULLS[outcome].std(ddof=1)),
        "n_null_draws": int(len(NULLS[outcome])),
    }
    print(
        f"{SCALING_OF[outcome].capitalize()} scaling ({UNIT[outcome]}): placebo sd {detection[outcome]['null_sd']:.4f}; "
        f"detection threshold {limits['threshold']:.4f}; minimum detectable effect {limits['mde']:.4f} from the empirical placebo distribution "
        f"and {limits['mde_normal']:.4f} from the normal approximation"
    )
    shares = [float(np.mean(np.abs(sign * limits["mde"] + NULLS[outcome]) > limits["threshold"])) for sign in (1.0, -1.0)]
    print(f"  Share of the placebo draws beyond the threshold for an effect of the minimum detectable size: {shares[0]:.3f} for a positive and {shares[1]:.3f} for a negative effect (power {POWER})")


def route_detection_probability(scn: str, route: str, outcome: str) -> float:
    """Probability that the observed effect of a route in one scenario exceeds the detection threshold.

    Parameters
    ----------
    scn : str
        Name of the scenario.
    route : str
        Key of ``ROUTE_DRAWS[scn]``.
    outcome : str
        ``pp`` or ``rel``.

    Returns
    -------
    float
        The share of the draws of the route beyond the threshold; for a route in ``TRUE_EFFECT_ROUTES`` each draw is
        first observed with an independent placebo draw.
    """
    noise = NULLS[outcome] if route in TRUE_EFFECT_ROUTES else None
    return impact.detection_probability(ROUTE_DRAWS[scn][route][outcome], detection[outcome]["threshold"], null_draws=noise)


detection_rows = []
for route in DETECTION_ROUTES:
    for outcome in ("pp", "rel"):
        median_effect = float(np.median(ROUTE_DRAWS[PRIMARY_SCENARIO][route][outcome]))
        detection_rows.append(
            {
                "route": route,
                "scaling": SCALING_OF[outcome],
                "median_effect": median_effect,
                "threshold": detection[outcome]["threshold"],
                "mde": detection[outcome]["mde"],
                "mde_normal": detection[outcome]["mde_normal"],
                "median_to_mde": median_effect / detection[outcome]["mde"],
                "placebo_noise_added": route in TRUE_EFFECT_ROUTES,
                "detection_probability": route_detection_probability(PRIMARY_SCENARIO, route, outcome),
            }
        )
detection_table = pd.DataFrame(detection_rows)
print(f"\nDetection by route, primary scenario (effects in percentage points of GDP for the absolute and percent of the counterfactual for the relative scaling)")
show(detection_table, 4)
scenario_detection_rows = []
for scn in SCEN_ORDER:
    for outcome in ("pp", "rel"):
        scenario_detection_rows.append(
            {
                "scenario": scn,
                "scaling": SCALING_OF[outcome],
                "median_effect": float(np.median(ROUTE_DRAWS[scn][decision.primary][outcome])),
                "median_to_mde": float(np.median(ROUTE_DRAWS[scn][decision.primary][outcome])) / detection[outcome]["mde"],
                "detection_probability": route_detection_probability(scn, decision.primary, outcome),
            }
        )
scenario_detection = pd.DataFrame(scenario_detection_rows)
print(f"\nDetection of the primary route {decision.primary} by scenario")
show(scenario_detection, 4)
print("\nThreshold and minimum detectable effect in US$ billion per year")
usd_rows = []
for year in BASELINE_YEARS:
    row = baseline.loc[year]
    for outcome in ("pp", "rel"):
        base = row[BASE_COLUMN[outcome]]
        usd_rows.append(
            {
                "baseline_year": year,
                "scaling": SCALING_OF[outcome],
                "threshold_usd_bn": detection[outcome]["threshold"] / 100.0 * base,
                "mde_usd_bn": detection[outcome]["mde"] / 100.0 * base,
                "mde_normal_usd_bn": detection[outcome]["mde_normal"] / 100.0 * base,
                "mde_share_of_receipts_pct": 100.0 * detection[outcome]["mde"] / 100.0 * base / row["receipts_usd_bn"],
                "mde_share_of_gdp_pct": 100.0 * detection[outcome]["mde"] / 100.0 * base / row["gdp_usd_bn"],
            }
        )
detection_usd = pd.DataFrame(usd_rows)
show(detection_usd, 3)

## Reference comparison with the claims of officials

The proposal file records claims that officials have made about the annual benefit of the park. They are shown only as a reference scale. The file states that no study is cited, so the claims are not estimates from a study and the analysis does not test them.

In [ ]:
claims = ctx.proposal.get("ministerial_claims") or {}
CLAIM_SHARE = claims.get("share_of_gdp_per_year_percent")
if claims:
    print("Claims recorded in the proposal file:", {k: v for k, v in claims.items() if k != "note"})
    print("Note in the file:", claims.get("note", "none"))
else:
    print("The proposal file records no claims.")
claim_rows = []
if CLAIM_SHARE is not None:
    for year in BASELINE_YEARS:
        row = baseline.loc[year]
        claim_usd = float(CLAIM_SHARE) / 100.0 * row["gdp_usd_bn"]
        claim_rows.append({"baseline_year": year, "claim_usd_bn": claim_usd, "claim_thb_bn": claim_usd * row["fx_thb_per_usd"], "claim_share_of_receipts_pct": 100.0 * claim_usd / row["receipts_usd_bn"]})
    claim_table = pd.DataFrame(claim_rows).set_index("baseline_year")
    print(f"\nA claim of {CLAIM_SHARE:g} percent of GDP a year in US$ billion and THB billion, and as a share of the receipts of the baseline year on the basis of the panel")
    show(claim_table, 3)
    primary_draws = ROUTE_DRAWS[PRIMARY_SCENARIO][decision.primary]
    for year in BASELINE_YEARS:
        row = baseline.loc[year]
        absolute_usd = primary_draws["pp"] / 100.0 * row["gdp_usd_bn"]
        relative_usd = primary_draws["rel"] / 100.0 * row["receipts_usd_bn"]
        claim_usd = float(CLAIM_SHARE) / 100.0 * row["gdp_usd_bn"]
        print(
            f"Baseline {year}, primary route {decision.primary}: share of predictive draws at or above the claim {100 * np.mean(absolute_usd >= claim_usd):.1f} percent "
            f"(absolute scaling) and {100 * np.mean(relative_usd >= claim_usd):.1f} percent (relative scaling)"
        )
    print("The claim is a reference scale only and has no study behind it.")

## Summary of the primary route

The summary file holds, for the primary route and each scaling and baseline year, the median and the 5th and 95th percentiles of the annual change in receipts in US$ billion and THB billion, as a share of the baseline receipts and of the baseline GDP, the central value that is quoted and the interval at the stated level with their kinds, together with the detection threshold, the minimum detectable effect from the empirical placebo distribution and from the normal approximation, and the detection probability. It also holds the support result of the target, the selection of the ambient estimator with its margin, the position of the target against the range of the episodes of the regression routes, the shrinkage of the ridge slopes, the percentiles of the sampling-only sensitivity of the importance route, and the same scaled summary for every other scenario of the proposal.

In [ ]:
primary_draws = ROUTE_DRAWS[PRIMARY_SCENARIO][decision.primary]


def route_block(scn: str, route: str, row: pd.Series) -> dict[str, object]:
    """Scaled summary of one route in one scenario for one baseline year.

    Parameters
    ----------
    scn : str
        Name of the scenario.
    route : str
        Key of ``ROUTE_DRAWS[scn]``.
    row : pandas.Series
        Baseline year with ``gdp_usd_bn``, ``receipts_usd_bn`` and ``fx_thb_per_usd``.

    Returns
    -------
    dict
        The output of ``scaled_summary`` with the entries ``interval`` (level, kind and bounds in US$ billion and THB
        billion) and ``central`` (kind and value in US$ billion and THB billion) in each scaling, and the entry
        ``baseline`` with the values of the baseline year.
    """
    block = scaled_summary(ROUTE_DRAWS[scn][route]["pp"], ROUTE_DRAWS[scn][route]["rel"], row)
    for outcome in ("pp", "rel"):
        scenario_entry = SCENARIOS[scn][RECORD_KEY[outcome]]
        record = scenario_entry["conformal_interval"] if route == "ot_importance" else None
        lower, upper, kind = interval_bounds(ROUTE_DRAWS[scn][route][outcome], route, record)
        centre, centre_kind = central_value(ROUTE_DRAWS[scn][route][outcome], route, scenario_entry["estimate"], record)
        factor = float(row[BASE_COLUMN[outcome]]) / 100.0
        fx = float(row["fx_thb_per_usd"])
        block[SCALING_OF[outcome]]["interval"] = {
            "level": LEVEL,
            "kind": kind,
            "usd_bn": {"lower": lower * factor, "upper": upper * factor},
            "thb_bn": {"lower": lower * factor * fx, "upper": upper * factor * fx},
        }
        block[SCALING_OF[outcome]]["central"] = {"kind": centre_kind, "usd_bn": centre * factor, "thb_bn": centre * factor * fx}
    block["baseline"] = {
        "gdp_usd_bn": float(row["gdp_usd_bn"]),
        "receipts_usd_bn": float(row["receipts_usd_bn"]),
        "receipts_usd_bn_baseline": float(row["receipts_usd_bn_baseline"]),
        "receipts_basis": str(row["receipts_basis"]),
        "fx_thb_per_usd": float(row["fx_thb_per_usd"]),
    }
    return block


primary_record = {outcome: SCENARIOS[PRIMARY_SCENARIO][RECORD_KEY[outcome]]["conformal_interval"] if decision.primary == "ot_importance" else None for outcome in ("pp", "rel")}
summary: dict[str, object] = {
    "primary_route": decision.primary,
    "primary_estimator": ESTIMATOR_OF[decision.primary],
    "predictive_method": PREDICTIVE_METHOD,
    "route_passed": {name: bool(flag) for name, flag in decision.passed.items()},
    "route_info": {key: value for key, value in decision.info.items()},
    "rule_sha256": RULE_SHA256,
    "support": {key: support[key] for key in ("supported", "ess", "max_source_share", "n_sources", "weighted_outside_share", "ess_ok", "range_ok", "reasons", "settings", "features")},
    "target": {
        "name": ctx.target_name,
        "scenario": PRIMARY_SCENARIO,
        "capex_thb_bn": float(target_record["capex_thb_bn"]),
        "cost_year": COST_YEAR,
        "fx_thb_per_usd": float(target_record["fx_thb_per_usd"]),
        "capex_usd_bn": float(target_record["capex_usd_bn"]),
        "capex_pct_gdp": TARGET["capex_pct_gdp"],
        "cost_year_alternative": COST_YEAR_ALTERNATIVE,
        "capex_pct_gdp_alternative": TARGET_ALTERNATIVE["capex_pct_gdp"],
        "receipts_pct_gdp": TARGET["receipts_pct_gdp"],
        "receipts_basis": target_record["receipts_basis"],
    },
    "effect": {
        "att_pp": quantile_dict(primary_draws["pp"]),
        "att_rel_pct": quantile_dict(primary_draws["rel"]),
        "interval": {
            RECORD_KEY[outcome]: dict(zip(("lower", "upper", "kind"), interval_bounds(primary_draws[outcome], decision.primary, primary_record[outcome])))
            for outcome in ("pp", "rel")
        },
    },
    "baseline_years": [int(y) for y in BASELINE_YEARS],
    "by_baseline_year": {},
    "detection": {},
    "ambient_selected": SELECTED,
    "ambient_selection": route_record["ambient_selection"],
    "regression_extrapolation": extrapolation_table.to_dict(orient="index"),
    "a3_shrinkage": a3_fit["shrinkage"].to_dict(orient="index"),
    "scenarios": {},
    "sensitivity": {
        "ot_importance_sampling": {
            "att_pp": quantile_dict(ROUTE_DRAWS[PRIMARY_SCENARIO]["ot_importance_sampling"]["pp"]),
            "att_rel_pct": quantile_dict(ROUTE_DRAWS[PRIMARY_SCENARIO]["ot_importance_sampling"]["rel"]),
        }
    },
}
for year in BASELINE_YEARS:
    summary["by_baseline_year"][str(year)] = route_block(PRIMARY_SCENARIO, decision.primary, baseline.loc[year])
for outcome in ("pp", "rel"):
    scaling = SCALING_OF[outcome]
    summary["detection"][scaling] = {
        "unit": UNIT[outcome],
        "threshold": detection[outcome]["threshold"],
        "minimum_detectable_effect": detection[outcome]["mde"],
        "mde_normal": detection[outcome]["mde_normal"],
        "detection_probability": route_detection_probability(PRIMARY_SCENARIO, decision.primary, outcome),
        "placebo_noise_added": decision.primary in TRUE_EFFECT_ROUTES,
        "null_sd": detection[outcome]["null_sd"],
        "null_rescale_factor": NULL_FACTOR if outcome == "pp" else 1.0,
        "alpha": ALPHA,
        "power": POWER,
    }
for scn in SCEN_ORDER:
    summary["scenarios"][scn] = {
        "capex_thb_bn": float(SCENARIOS[scn]["capex_thb_bn"]),
        "capex_pct_gdp": float(SCENARIOS[scn]["capex_pct_gdp"]),
        "importance_route_supported": bool(SCENARIOS[scn]["support"]["supported"]),
        "importance_route_reasons": list(SCENARIOS[scn]["support"]["reasons"]),
        "by_baseline_year": {str(year): route_block(scn, decision.primary, baseline.loc[year]) for year in BASELINE_YEARS},
        "detection_probability": {
            SCALING_OF[outcome]: route_detection_probability(scn, decision.primary, outcome) for outcome in ("pp", "rel")
        },
    }
if CLAIM_SHARE is not None:
    summary["claim_reference"] = {"share_of_gdp_per_year_percent": float(CLAIM_SHARE), "note": claims.get("note"), "status": "reference only"}
saved_summary = ctx.save_json(summary, "thailand_summary")

## Figures

The first figure shows the annual change in receipts in US$ billion for every route and both scalings, for each baseline year of the primary scenario. The thick bar is the 50 percent interval of the draws, the thin bar is the interval at the stated level, and the marker is the central value: the transported effect for the importance route and the median of the draws for the other routes. The thin bar is the conformal interval for the importance route, which is the interval for the effect observed in a new study; it is the percentile interval of the bootstrap draws for the sampling-only row, which is the sampling uncertainty of the transported average effect; and it is the percentile interval of the predictive draws for the ambient rows. The primary route has filled markers and the claim, when the proposal file states one, is the dashed vertical line.

The second figure shows the case effects against the cost share of GDP with the fitted scaling law and its prediction for the target economy, and the central value and the interval of the primary route at the cost share of the target. The third figure shows the distribution of the importance route for the primary scenario: the predictive draws rescaled to the conformal interval, the bootstrap draws of the transported average effect, and the transported placebo null, with the transported effect, the conformal interval and the percentile interval of the bootstrap draws marked.

In [ ]:
ROUTE_LABEL = {
    "ot_importance": "Importance-weighted transport, predictive",
    "ot_importance_sampling": "Importance-weighted transport, sampling only",
    "ambient_A1": "Ambient A1, pooled hierarchical",
    "ambient_A2": "Ambient A2, scaling law in cost",
    "ambient_A3": "Ambient A3, meta-regression",
}
primary_rows = comparison[comparison["scenario"] == PRIMARY_SCENARIO]
ROUTE_ORDER = [route for route in ROUTE_LABEL if route in set(primary_rows["route"])]
PRIMARY_KEY = "ot_importance" if decision.primary == "ot_importance" else f"ambient_{SELECTED}"
fig, axes = plt.subplots(1, len(BASELINE_YEARS), figsize=(6.2 * len(BASELINE_YEARS), 0.5 * 2 * len(ROUTE_ORDER) + 1.2), sharey=True)
axes = np.atleast_1d(axes)
labels = []
for route in ROUTE_ORDER:
    for scaling in ("absolute", "relative"):
        labels.append(f"{ROUTE_LABEL[route]}, {scaling}" + (" (primary)" if route == PRIMARY_KEY else ""))
ypos = np.arange(len(labels))[::-1]
for ax, year in zip(axes, BASELINE_YEARS):
    k = 0
    for route in ROUTE_ORDER:
        for scaling, marker in (("absolute", "o"), ("relative", "s")):
            r = primary_rows[(primary_rows["route"] == route) & (primary_rows["baseline_year"] == year) & (primary_rows["scaling"] == scaling)].iloc[0]
            y = ypos[k]
            ax.hlines(y, r["q25"], r["q75"], color="0.62", linewidth=7.0)
            ax.hlines(y, r["interval_lower"], r["interval_upper"], color="0.0", linewidth=1.3)
            ax.plot(
                r["central"], y, marker=marker, markersize=8, linestyle="none", markeredgecolor="0.0",
                markerfacecolor="0.0" if route == PRIMARY_KEY else "white", zorder=4,
            )
            k += 1
    ax.axvline(0.0, color="0.0", linewidth=0.8)
    if CLAIM_SHARE is not None:
        claim_usd = float(CLAIM_SHARE) / 100.0 * baseline.loc[year, "gdp_usd_bn"]
        ax.axvline(claim_usd, color="0.2", linestyle="--", linewidth=1.3)
    ax.set_title(f"Baseline {year}")
    ax.set_xlabel("Annual change in receipts (US$ billion per year)")
    ax.set_yticks(ypos)
    ax.set_yticklabels(labels, fontsize=8)
    ax.grid(axis="y", visible=False)
axes[0].set_ylabel("Route and scaling")
handles = [
    Line2D([], [], color="0.62", linewidth=7.0, label="50 percent interval of the draws"),
    Line2D([], [], color="0.0", linewidth=1.3, label=f"{LEVEL_TEXT} interval (conformal for the importance route, percentile of the draws otherwise)"),
    Line2D([], [], color="0.0", marker="o", markerfacecolor="white", linestyle="none", label="Central value, absolute scaling"),
    Line2D([], [], color="0.0", marker="s", markerfacecolor="white", linestyle="none", label="Central value, relative scaling"),
    Line2D([], [], color="0.0", marker="o", markerfacecolor="0.0", linestyle="none", label="Primary route (filled)"),
]
if CLAIM_SHARE is not None:
    handles.append(Line2D([], [], color="0.2", linestyle="--", linewidth=1.3, label=f"Claim of {CLAIM_SHARE:g} percent of GDP a year (reference only)"))
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=2, fontsize=8)
fig.suptitle("Annual change in international tourism receipts by route", y=1.0)
fig.tight_layout()
figure_path = ctx.save_figure(fig, "impact_routes.png")

In [ ]:
assert "A2" in AVAILABLE, "the scaling-law figure needs the prediction interval of A2"
x_case = AMB["capex_pct_gdp"].to_numpy(dtype=float)
y_case = AMB["att_pp"].to_numpy(dtype=float)
se_case = AMB["placebo_sd_pp"].to_numpy(dtype=float)
a2 = AMBIENT["A2"]["pp"]
t_level = float(stats.t.ppf(0.5 + LEVEL / 2.0, a2_fit["df"]))
PRIMARY_LABEL = "importance-weighted transport" if decision.primary == "ot_importance" else f"ambient {SELECTED}"
primary_pp = ROUTE_DRAWS[PRIMARY_SCENARIO][decision.primary]["pp"]
primary_lower, primary_upper, primary_kind = interval_bounds(primary_pp, decision.primary, primary_record["pp"])
primary_centre, primary_centre_kind = central_value(primary_pp, decision.primary, SCENARIOS[PRIMARY_SCENARIO]["att_pp"]["estimate"], primary_record["pp"])


def band(x: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Fitted scaling law and the edges of its prediction interval at the sizes ``x``."""
    centre = a2_fit["beta"] * x
    half = t_level * np.sqrt(a2_fit["tau2"] + x**2 * a2_fit["se_beta"] ** 2)
    return centre, centre - half, centre + half


fig, axes = plt.subplots(1, 2, figsize=(11.0, 5.2))
x_ranges = (
    ("Range of the case costs", np.linspace(0.0, 1.08 * x_case.max(), 200)),
    ("Range extended to the target cost", np.linspace(0.0, 1.08 * max(x_case.max(), TARGET["capex_pct_gdp"]), 200)),
)
for ax, (title, grid) in zip(axes, x_ranges):
    centre, lower, upper = band(grid)
    ax.fill_between(grid, lower, upper, color="0.88", label=f"{LEVEL_TEXT} prediction interval of the scaling law")
    ax.plot(grid, centre, color="0.0", linestyle="-", linewidth=1.5, label="Fitted scaling law through the origin")
    ax.errorbar(x_case, y_case, yerr=se_case, fmt="o", color="0.0", markerfacecolor="white", markersize=6, capsize=2, linewidth=0.9, label="Case effect (bars: one placebo standard deviation)", zorder=3)
    offset = 0.012 * float(grid.max())
    ax.errorbar(
        [TARGET["capex_pct_gdp"] - offset], [a2["mean"]], yerr=[[a2["mean"] - a2["lo"]], [a2["hi"] - a2["mean"]]],
        fmt="D", color="0.35", markersize=8, capsize=4, linewidth=1.6, label=f"Target, scaling law A2: prediction and {LEVEL_TEXT} interval", zorder=4,
    )
    ax.errorbar(
        [TARGET["capex_pct_gdp"] + offset], [primary_centre], yerr=[[primary_centre - primary_lower], [primary_upper - primary_centre]],
        fmt="s", color="0.0", markersize=8, capsize=4, linewidth=1.6, label=f"Target, primary route ({PRIMARY_LABEL}): {primary_centre_kind} and {LEVEL_TEXT} interval", zorder=5,
    )
    ax.axhline(0.0, color="0.0", linewidth=0.8)
    ax.set_xlim(left=0.0)
    ax.set_xlabel("Cost of the park as a share of GDP (percent of GDP)")
    ax.set_ylabel("Effect on receipts (percentage points of GDP)")
    ax.set_title(title)
handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=2, fontsize=8)
fig.suptitle("Case effects, the scaling law in the cost share of GDP and the primary route", y=1.0)
fig.tight_layout()
figure_path = ctx.save_figure(fig, "impact_scaling_law.png")

In [ ]:
fig, (ax_pp, ax_rel) = plt.subplots(1, 2, figsize=(11.0, 4.8))
for outcome, ax, null_o, unit_label, title in (
    ("pp", ax_pp, NULLS["pp"], "Mean annual effect over five years (percentage points of GDP)", "Effect in percentage points of GDP"),
    ("rel", ax_rel, NULLS["rel"], "Mean annual effect over five years (percent of counterfactual receipts)", "Effect in percent of the counterfactual"),
):
    draws = ROUTE_DRAWS[PRIMARY_SCENARIO]["ot_importance"][outcome]
    sampling = ROUTE_DRAWS[PRIMARY_SCENARIO]["ot_importance_sampling"][outcome]
    lo = min(np.percentile(draws, 0.5), np.percentile(null_o, 0.5), np.percentile(sampling, 0.5))
    hi = max(np.percentile(draws, 99.5), np.percentile(null_o, 99.5), np.percentile(sampling, 99.5))
    bins = np.linspace(lo, hi, 41)
    record = OT_RECORD[outcome]
    ax.hist(draws, bins=bins, density=True, color="0.75", edgecolor="0.3", label="Predictive draws (rescaled to the conformal interval)")
    ax.hist(sampling, bins=bins, density=True, histtype="step", color="0.35", linewidth=1.4, linestyle=":", label="Bootstrap draws of the transported average effect")
    ax.hist(
        null_o, bins=bins, density=True, histtype="step", color="0.0", linewidth=1.4, linestyle="--",
        label="Transported placebo null" + (" (on the receipts share of the target)" if outcome == "pp" else ""),
    )
    ax.axvline(record["estimate"], color="0.0", linestyle="-", linewidth=1.6, label="Transported effect")
    interval = record["conformal_interval"]
    if interval["status"] == "ok":
        ax.axvline(interval["lower"], color="0.25", linestyle="-.", linewidth=1.2, label=f"Conformal {LEVEL_TEXT} interval (new study at the target)")
        ax.axvline(interval["upper"], color="0.25", linestyle="-.", linewidth=1.2)
    boot = record["bootstrap_percentiles"]
    ax.axvline(boot["p5"], color="0.5", linestyle=":", linewidth=1.6, label="5th and 95th percentiles of the bootstrap draws (sampling interval)")
    ax.axvline(boot["p95"], color="0.5", linestyle=":", linewidth=1.6)
    ax.set_xlabel(unit_label)
    ax.set_ylabel("Density")
    ax.set_title(title)
fig.suptitle("Importance route: distributions and intervals of the transported effect", y=1.02)
fig.tight_layout()
handles, labels_ = ax_pp.get_legend_handles_labels()
fig.legend(handles, labels_, loc="upper center", bbox_to_anchor=(0.5, -0.01), ncol=2, fontsize=8)
figure_path = ctx.save_figure(fig, "ot_predictive_distribution.png")

## Conclusions

The sentences below are produced from the results above.

In [ ]:
CRITERION_NAME = {
    "importance_usable": "importance model usable",
    "loco_rmse": "validation RMSE criterion",
    "enough_cases": "enough validated episodes",
    "target_supported": "target inside the support of the sources",
}
primary_label = decision.primary if decision.primary == "ot_importance" else f"ambient ({SELECTED}, {DESCRIPTION[SELECTED]})"
other_key = "ambient" if decision.primary == "ot_importance" else "ot_importance"
print(
    f"The rule selects the {primary_label} route as primary (hash of the rule {RULE_SHA256[:12]}). Criteria met: "
    + ", ".join(f"{CRITERION_NAME[name]} {'yes' if flag else 'no'}" for name, flag in decision.passed.items())
    + f". The validation relearned the importance weights in {n_folds - n_uniform_folds} of {n_folds} folds."
)
if not support["supported"]:
    print("The support check reports that the target is not supported by the sources: " + " ".join(support["reasons"]))
for year in BASELINE_YEARS:
    block = summary["by_baseline_year"][str(year)]
    for scaling in ("absolute", "relative"):
        b = block[scaling]
        centre = b["central"]
        print(
            f"With the baseline of {year} and the {scaling} scaling, the primary route puts the annual change in international tourism receipts at US$ {centre['usd_bn']:.2f} billion "
            f"(THB {centre['thb_bn']:.0f} billion; {centre['kind']}), with a {LEVEL_TEXT} interval from US$ {b['interval']['usd_bn']['lower']:.2f} to {b['interval']['usd_bn']['upper']:.2f} billion "
            f"({b['interval']['kind']}). That is {100 * centre['usd_bn'] / block['baseline']['receipts_usd_bn']:.1f} percent of the receipts and {100 * centre['usd_bn'] / block['baseline']['gdp_usd_bn']:.2f} percent of the GDP of {year}."
        )
for outcome in ("pp", "rel"):
    scaling = SCALING_OF[outcome]
    d = summary["detection"][scaling]
    print(
        f"On the {scaling} scaling the transported placebo draws give a detection threshold of {d['threshold']:.3f} and a minimum detectable effect of {d['minimum_detectable_effect']:.3f} {UNIT[outcome]} "
        f"({d['mde_normal']:.3f} with the normal approximation); the probability that the observed effect of the primary route exceeds the threshold is {100 * d['detection_probability']:.1f} percent."
    )
print(
    f"The null in percentage points of GDP was multiplied by {NULL_FACTOR:.3f}, the receipts share of the target over the receipts share of the sources behind it."
)
other = comparison[(comparison["scenario"] == PRIMARY_SCENARIO) & (comparison["route"] == other_key) & (comparison["baseline_year"] == BASELINE_YEARS[0]) & (comparison["scaling"] == "absolute")].iloc[0]
primary_row = comparison[(comparison["scenario"] == PRIMARY_SCENARIO) & (comparison["primary"]) & (comparison["baseline_year"] == BASELINE_YEARS[0]) & (comparison["scaling"] == "absolute")].iloc[0]
sampling_row = comparison[(comparison["scenario"] == PRIMARY_SCENARIO) & (comparison["route"] == "ot_importance_sampling") & (comparison["baseline_year"] == BASELINE_YEARS[0]) & (comparison["scaling"] == "absolute")].iloc[0]
print(
    f"For comparison, the {other_key} route gives US$ {other['central']:.2f} billion ({other['central_kind']}; {LEVEL_TEXT} interval {other['interval_lower']:.2f} to {other['interval_upper']:.2f}) on the absolute scaling "
    f"with the baseline of {BASELINE_YEARS[0]}, against US$ {primary_row['central']:.2f} billion for the primary route."
)
print(
    f"The importance route takes its predictive distribution from the leave-one-economy-out errors of {PREDICTIVE_METHOD}, and its interval is the conformal interval. The sampling distribution of the transported average effect alone gives a {LEVEL_TEXT} interval "
    f"from US$ {sampling_row['interval_lower']:.2f} to {sampling_row['interval_upper']:.2f} billion on the absolute scaling with the baseline of {BASELINE_YEARS[0]}."
)
print(
    f"The ambient estimator {SELECTED} was selected; the selection is close: {selection['close']}; the runner up {selection['runner_up']} has an RMSE that differs from it by {100 * selection['margin']:+.1f} percent."
)
print(
    f"The cost of the proposal is {TARGET['capex_pct_gdp']:.2f} percent of GDP, {TARGET['capex_pct_gdp'] / AMB['capex_pct_gdp'].max():.2f} times the largest cost share among the {len(AMB)} primary episodes; "
    f"the regression routes extrapolate on {EXTRAPOLATES or 'no covariate'}."
)
if CLAIM_SHARE is not None:
    claim_first = claim_table.loc[BASELINE_YEARS[0]]
    print(
        f"The claim of {CLAIM_SHARE:g} percent of GDP a year corresponds to US$ {claim_first['claim_usd_bn']:.2f} billion with the baseline of {BASELINE_YEARS[0]}. "
        f"It is shown as a reference only; no study stands behind it."
    )
scenario_lines = comparison[(comparison["primary"]) & (comparison["baseline_year"] == BASELINE_YEARS[0]) & (comparison["scaling"] == "absolute")]
for _, r in scenario_lines.iterrows():
    print(
        f"Scenario {r['scenario']} (THB {r['capex_thb_bn']:g} billion, {r['capex_pct_gdp']:.2f} percent of GDP): primary route US$ {r['central']:.2f} billion ({r['central_kind']}) on the absolute scaling with the baseline of {BASELINE_YEARS[0]}; "
        f"importance route supported by the sources: {r['importance_route_supported']}."
    )

## Checks

In [ ]:
assert decision.primary in ("ot_importance", "ambient")
assert list(decision.passed) == CRITERIA
EXPECTED_ROUTES = {"ot_importance", "ot_importance_sampling", "ambient", *(f"ambient_{name}" for name in AVAILABLE)}
for scn in SCEN_ORDER:
    part = comparison[comparison["scenario"] == scn]
    expected = EXPECTED_ROUTES if scn == PRIMARY_SCENARIO else EXPECTED_ROUTES - {"ot_importance_sampling"}
    assert set(part["route"]) == expected, scn
    assert len(part) == len(expected) * len(BASELINE_YEARS) * 2
    assert part.groupby("route")["primary"].all().loc[decision.primary] and not part.groupby("route")["primary"].any().drop(decision.primary).any()
assert (comparison["q5"] <= comparison["q50"]).all() and (comparison["q50"] <= comparison["q95"]).all()
assert (comparison["interval_lower"] <= comparison["interval_upper"]).all()
assert ((comparison["interval_lower"] <= comparison["central"] + 1e-9) & (comparison["central"] <= comparison["interval_upper"] + 1e-9)).all()
assert set(comparison.loc[comparison["route"] != "ot_importance", "central_kind"]) == {"median of the draws"}
assert np.isfinite(comparison[["q5", "q25", "q50", "q75", "q95", "interval_lower", "interval_upper", "median_share_receipts_pct", "median_share_gdp_pct"]].to_numpy()).all()
assert set(loeo.index) == set(ESTIMATORS) and np.isfinite(loeo["rmse"]).all() and loeo["selected"].sum() == 1 and SELECTED in AVAILABLE
assert all(np.isfinite(ROUTE_DRAWS[scn][route][o]).all() and len(ROUTE_DRAWS[scn][route][o]) > 100 for scn in ROUTE_DRAWS for route in ROUTE_DRAWS[scn] for o in ("pp", "rel"))

# The route rule recomputed from the saved validation summary and the support result
rmse = loco_summary["rmse"]
rule_met = (
    IMPORTANCE_USABLE
    and rmse["ot_weighted"] <= rules["rmse_ratio_equal"] * rmse["equal"]
    and rmse["ot_weighted"] <= rules["rmse_ratio_uniform"] * rmse["ot_uniform"]
    and n_validated >= rules["min_cases"]
    and bool(support["supported"])
)
assert (decision.primary == "ot_importance") == bool(rule_met)
assert 0 <= n_uniform_folds <= n_folds == n_validated_economies
assert np.allclose(fold_weights.groupby("held_out_group")["weight"].sum(), 1.0)
assert decision.info["rmse_ratio_to_best_ambient"] > 0 and decision.info["n_ambient"] == len(AMB)
assert RULE_SHA256 == hashlib.sha256(json.dumps(RULE, sort_keys=True).encode("utf-8")).hexdigest()

# The ambient selection recomputed
ratios_recomputed = loeo["rmse"] / loeo.loc["A1", "rmse"]
qualifying_recomputed = [name for name in ("A2", "A3") if ratios_recomputed[name] <= SELECTION_RATIO * (1.0 + 1e-12)]
expected_selected = "A1" if not qualifying_recomputed else min(qualifying_recomputed, key=lambda n: loeo.loc[n, "rmse"])
assert SELECTED == expected_selected and set(selection["qualifying"]) == set(qualifying_recomputed)

# The importance route uses the predictive draws of the previous notebook
for outcome, key in (("pp", "att_pp"), ("rel", "att_rel_pct")):
    stored = transport_record[key]["predictive_percentiles"]
    draws = ROUTE_DRAWS[PRIMARY_SCENARIO]["ot_importance"][outcome]
    assert np.allclose([stored["p5"], stored["p50"], stored["p95"]], np.quantile(draws, [0.05, 0.5, 0.95]))
    interval = transport_record[key]["conformal_interval"]
    if interval["status"] == "ok":
        assert interval["lower"] <= transport_record[key]["estimate"] <= interval["upper"]
assert transport_record["att_pp"]["predictive_method"] == PREDICTIVE_METHOD == transport_record["predictive"]["method"]

# The null: the absolute null is the saved null times the receipts-share factor, and the relative null is unchanged
assert np.allclose(NULLS["pp"], null_draws["null_pp"].to_numpy(dtype=float) * (TARGET["receipts_pct_gdp"] / SOURCE_SHARE))
assert np.array_equal(NULLS["rel"], null_draws["null_rel_pct"].to_numpy(dtype=float))
manual_share = float((usage_table["mass_sent"] * source_shares).sum() / usage_table["mass_sent"].sum())
assert np.isclose(SOURCE_SHARE, manual_share)

# Detection: threshold, minimum detectable effect and probability recomputed directly from the placebo draws
for outcome in ("pp", "rel"):
    null = NULLS[outcome]
    threshold = float(np.quantile(np.abs(null), 1.0 - ALPHA))
    assert np.isclose(detection[outcome]["threshold"], threshold)
    assert detection[outcome]["mde"] >= detection[outcome]["threshold"] > 0 and np.isfinite(detection[outcome]["mde_normal"]) and detection[outcome]["mde_normal"] > 0
    atom = float(np.unique(null, return_counts=True)[1].max()) / len(null)
    for sign in (1.0, -1.0):
        assert np.mean(np.abs(sign * detection[outcome]["mde"] + null) > threshold) >= POWER - atom
    sample = slice(0, 500)
    for route in DETECTION_ROUTES:
        draws = ROUTE_DRAWS[PRIMARY_SCENARIO][route][outcome][sample]
        shifted = draws[:, None] + null[None, sample] if route in TRUE_EFFECT_ROUTES else draws[:, None]
        assert np.isclose(
            impact.detection_probability(draws, threshold, null_draws=null[sample] if route in TRUE_EFFECT_ROUTES else None),
            np.mean(np.abs(shifted) > threshold),
        )
assert all(0.0 <= row["detection_probability"] <= 1.0 for row in detection_rows) and {row["route"] for row in detection_rows} == set(DETECTION_ROUTES)

# The saved files
for name in ("route_decision.json", "thailand_summary.json"):
    assert (ctx.results_dir / name).is_file(), name
for name in ("thailand_impact", "ambient_loeo"):
    assert ctx.table_path(name).is_file(), name
saved_decision = ctx.read_json("route_decision")
assert saved_decision["primary"] == decision.primary and set(saved_decision) >= {"primary", "passed", "details", "inputs", "ambient_selection", "info"}
assert len(saved_decision["passed"]) == 4 and saved_decision["rule"]["sha256"] == RULE_SHA256
assert saved_decision["inputs"]["n_validated_episodes"] == n_validated and "leave-one-economy-out" in saved_decision["details"]
saved_summary_check = ctx.read_json("thailand_summary")
assert saved_summary_check["primary_route"] == decision.primary and set(saved_summary_check["by_baseline_year"]) == {str(y) for y in BASELINE_YEARS}
assert saved_summary_check["predictive_method"] == PREDICTIVE_METHOD and set(saved_summary_check["sensitivity"]) == {"ot_importance_sampling"}
assert set(saved_summary_check["scenarios"]) == set(SCEN_ORDER) and saved_summary_check["support"]["supported"] == support["supported"]
assert all("mde_normal" in saved_summary_check["detection"][scaling] for scaling in ("absolute", "relative"))
saved_impact_check = pd.read_csv(ctx.table_path("thailand_impact"))
assert saved_impact_check["primary"].dtype == bool and set(saved_impact_check["route"]) == EXPECTED_ROUTES and set(saved_impact_check["scenario"]) == set(SCEN_ORDER)
print("Checks passed.")
print(f"Elapsed time of the notebook: {time.perf_counter() - NOTEBOOK_START:.0f} seconds")

## Scope and limits

The estimates are conditional on a park of the stated cost opening. They are mean annual effects on international tourism receipts over the first five years after the opening, in percentage points of GDP and in percent of the counterfactual receipts, scaled to US dollars and baht with the baseline of the year named in each table. The importance route moves the estimated effects of earlier openings to the feature cloud of Thailand; it does not estimate the effect of a Thai park from Thai data, and the estimates are not the total economic impact of a park, which would add domestic visitors, employment, tax revenue and displacement of other spending.

The route rule compares leave-one-economy-out errors in which the importance weights are relearned inside every fold from the training economies, so the validation is not optimistic about the learning of the weights, and a fold whose weights could not be fitted is counted and uses uniform weights. The folds use the light configuration of the importance estimator, so the validation does not judge the weights of the full configuration exactly. The comparison is a point rule on the RMSE; the bootstrap intervals of the RMSE ratios in the previous notebook describe the cases at hand and are not a test of the rule. The rule counts episodes, while the independent units are economies. The comparison with the ambient estimators and with the nearest-neighbour predictors is reported and is not a criterion, and the ratio compares like with like only when the transport and the ambient estimators are validated on the same economies, which the printed sample sizes show.

The importance route quotes the conformal interval, which has a finite-sample coverage guarantee for the effect observed in a new study when the validation errors are exchangeable with the error at the target. Errors of episodes of one economy are dependent, the training sets of the folds overlap, and the validation errors are errors at the cases' own features, so the guarantee is approximate and holds for a target that resembles the cases. A transport of past effects cannot exceed the largest effect among its sources, so for a target that the support check reports as not supported the importance route is bounded by the data however far out the target lies. The sampling distribution of the transported average effect, shown as a sensitivity, describes the mean effect and not the effect of one new opening. The percentiles of the saved predictive draws are descriptive and carry no coverage guarantee.

The case effects come from country-level synthetic controls with few pre-opening years, and the pandemic years are excluded from every estimation window. The proposal is not a commitment: the sources reviewed record no signed agreement, approved budget, named operator or site allocation. The cost in US dollars is derived from the cost in baht with the exchange rate and the GDP of one year, and the dollar costs of the earlier cases are derived in the same way, so every cost is an estimate; the alternative cost year shows how much the cost share depends on the year. The receipts of the baseline are put on the basis of the panel series, and for the years without a panel value they rest on the mean ratio of the panel to the baseline over the years in which both exist.

Cases in one economy are not independent. The validation of the ambient estimators leaves out whole economies, and the printed number of economies, not of episodes, is the effective number of independent observations. When the cost of the proposal as a share of GDP or the receipts share lies beyond the range of the cases, the scaling law and the meta-regression extrapolate, the printed position and ratio to the largest case report how far, and the ridge penalty of the meta-regression shrinks the slopes, which moves an extrapolated prediction. The selection among the ambient estimators is reported with its margin, and a close selection means that another estimator could have been chosen with a small change in the data. The detection analysis takes the placebo distribution of the transport method as the noise of an observed effect, with the absolute null put on the receipts share of the target; the draws of the ambient routes are observed with a draw from it, and the draws of the importance route already carry the prediction errors. The null belongs to the primary scenario and is used for every scenario. The claims of officials are a reference scale only, because no study stands behind them.